# Spring Petclinic 1.5.x Migration Research Agent
This notebook sets up a LangGraph-based Research Agent.
Its goal is to parse `spring-petclinic-1.5.x/pom.xml`, identify 3rd-party libraries, check their compatibility against target JDK (17) and Spring Boot versions, and output a mapping manifest for the upgrade.


In [1]:
import os
from dotenv import load_dotenv
from uuid import uuid4
import xml.etree.ElementTree as ET
import requests
from typing import List, Dict, Any
from getpass import getpass



from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

# Make sure you have the API keys set or prompt for them
load_dotenv(override=True)


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# We use the fast and capable gpt-4o or gpt-4o-mini here
llm = ChatOpenAI(model="gpt-4o", temperature=0)


### Agent Tools
Here we define the custom tools that the agent will use:
1. `parse_pom_xml`: Extracts 3rd party dependencies.
2. `search_maven_central`: Looks up latest library versions.
3. `search_compatibility_info`: A Tavily-backed tool to find compatibility constraints online.

In [2]:
@tool
def parse_pom_xml(filepath: str) -> List[Dict[str, str]]:
    """Parses a Maven pom.xml and returns a list of dependencies with their groupId, artifactId, and versions.
    Filters out the org.springframework.* dependencies to focus on 3rd-party libraries.
    """
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        namespace = {'mvn': 'http://maven.apache.org/POM/4.0.0'}
        
        dependencies = []
        # Find all dependencies
        for dep in root.findall('.//mvn:dependency', namespace):
            group_id = dep.find('mvn:groupId', namespace)
            artifact_id = dep.find('mvn:artifactId', namespace)
            version = dep.find('mvn:version', namespace)
            
            g_id = group_id.text if group_id is not None else "UNKNOWN"
            a_id = artifact_id.text if artifact_id is not None else "UNKNOWN"
            v = version.text if version is not None else "MANAGED"
            
            # Simple filter to exclude core spring libraries
            if not g_id.startswith('org.springframework'):
                dependencies.append({"groupId": g_id, "artifactId": a_id, "version": v})
                
        return dependencies
    except Exception as e:
        return [{"error": str(e)}]

@tool
def search_maven_central(group_id: str, artifact_id: str) -> str:
    """Queries the Maven Central public API to find the latest versions of an artifact.
    Use this to see what versions exist.
    """
    url = f"https://search.maven.org/solrsearch/select?q=g:{group_id}+AND+a:{artifact_id}&rows=5&core=gav&wt=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        docs = data.get('response', {}).get('docs', [])
        versions = [doc.get('v') for doc in docs]
        if versions:
            return f"Available versions for {group_id}:{artifact_id}: {', '.join(versions)}"
        return f"No versions found for {group_id}:{artifact_id}."
    return f"Error querying Maven Central: {response.status_code}"

# Define our Tavily search tool for RAG and Spring compatibility web lookup
search_compatibility_info = TavilySearchResults(max_results=3, description="Use this tool to search the internet for Spring Boot and JDK compatibility matrices for specific libraries.")

# List of tools for the agent
tools = [parse_pom_xml, search_maven_central, search_compatibility_info]


/var/folders/j4/9sv02n0d3hxdb2cdl4cn6_1c0000gn/T/ipykernel_13354/776584977.py:47: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_compatibility_info = TavilySearchResults(max_results=3, description="Use this tool to search the internet for Spring Boot and JDK compatibility matrices for specific libraries.")


### Agent Graph Definition
We build a standard ReAct LangGraph using our configured tools and LLM.

In [3]:
# System prompt directing the agent's behavior
system_message = """You are an expert Java developer and a Research Agent specializing in migrating legacy Spring Boot applications.
Your task is to analyze a given project's dependencies and output a mapped manifest suggesting the minimum viable version updates to support a new target JDK and Spring framework.

Process:
1. Use `parse_pom_xml` to extract dependencies (only 3rd party are parsed).
2. For each dependency, use `search_maven_central` to check newest versions and `search_compatibility_info` to ensure the new version is compatible with the target JDK and Spring Boot version.
3. For the output, you MUST return a strict JSON object (not Markdown, just raw JSON or within a JSON codeblock) containing a `dependencies` array. Each object in the array should have `groupId`, `artifactId`, `currentVersion`, `recommendedVersion`, and `reasoning`.

Important Note: The user might ask for 'Spring Boot 5', but please know that the latest major version is Spring Boot 3.x (wrapping Spring Framework 6.x). If they specify 'Spring Boot 5', they likely mean 'Spring Framework 5' (which corresponds to Spring Boot 2.x) or they made a typo and meant Spring Boot 3. Adjust your web searches appropriately, but deliver the requested research smoothly!
"""

# Create the LangGraph tool-calling ReAct agent
agent = create_react_agent(llm, tools, prompt=system_message)


### Run and Test the Agent
Execute this block to run the agent.

In [4]:
import json
import re

query = "Parse the spring-petclinic-1.5.x/pom.xml, identify every 3rd-party library, and determine the minimum viable version that supports JDK 17 and Spring Boot 3.x (Spring Framework 6.x). Please output strictly as a JSON report."

messages = agent.invoke({"messages": [("user", query)]})
final_output = messages["messages"][-1].content

# Use regex to extract json block if wrapped in markdown
match = re.search(r"```(?:json)?(.*?)```", final_output, re.DOTALL)
json_str = match.group(1).strip() if match else final_output.strip()

try:
    report_data = json.loads(json_str)
    with open("migration_report.json", "w") as f:
        json.dump(report_data, f, indent=4)
    print("Successfully saved report to migration_report.json")
except json.JSONDecodeError as e:
    print("Agent did not return valid JSON! Raw output:")
    print(final_output)


Successfully saved report to migration_report.json
